# Knowledge Base Telegram Bot

Этот блокнот реализует Telegram-бота на aiogram3, который отвечает на вопросы о зимних Олимпийских играх 2022, используя метод Search-Ask с эмбеддингами OpenAI и GPT.

## Как это работает
1. Загружается готовая база знаний из облачного хранилища (CSV с текстом + эмбеддинги).
2. Вопрос пользователя преобразуется в эмбеддинг, находятся наиболее релевантные секции по косинусному расстоянию.
3. Релевантные секции добавляются как контекст в промпт GPT.
4. Ответ GPT возвращается пользователю.

## Команды
- `/start` - Приветственное сообщение
- `/help` - Информация о базе знаний (тематика, количество записей, пример запроса)
- Любое текстовое сообщение - Поиск и ответ из базы знаний

In [1]:
# Установка зависимостей
!pip install aiogram openai pandas scipy nest-asyncio tiktoken -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 804.4/804.4 kB 13.9 MB/s eta 0:00:00


In [4]:
# Импорты
import nest_asyncio
nest_asyncio.apply()

import asyncio
import logging
import ast
import os
import pandas as pd
from scipy import spatial
import tiktoken
import requests
import zipfile
import io
from openai import OpenAI
from aiogram import Bot, Dispatcher, types, F
from aiogram.filters.command import Command
from google.colab import userdata

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

In [29]:
# Настройка API
# Используем секреты Colab
M_TOKEN = userdata.get('M_TOKEN')
B_TOKEN = userdata.get('B_TOKEN')
BASE_URL = userdata.get('BASE_URL')

openai_client = OpenAI(
    base_url=BASE_URL,
    api_key=M_TOKEN,
)

# Модели
GPT_MODEL = "openai/gpt-4o-mini"
EMBEDDING_MODEL = "openai/text-embedding-3-small"

print("API настроен")

API настроен


In [9]:
# Загрузка базы знаний из репозитория GitHub

ZIP_URL = "https://github.com/z123p2/knowledge-base-olympics-2022-bot/raw/main/data/winter_olympics_2022_v3_small.zip"
CSV_PATH = "/content/winter_olympics_2022_v3_small.csv"

try:
    if not os.path.exists(CSV_PATH):
        print("Скачиваю базу знаний из репозитория...")
        r = requests.get(ZIP_URL)
        with zipfile.ZipFile(io.BytesIO(r.content)) as z:
            z.extractall("/content")
        print("Распаковано.")

    df = pd.read_csv(CSV_PATH)
    df['embedding'] = df['embedding'].apply(ast.literal_eval)
    print(f"База знаний успешно загружена.")
    print(f"Тематика: 2022 Winter Olympics")
    print(f"Количество записей: {len(df)}")
except Exception as e:
    print(f"Не удалось загрузить базу знаний: {e}")
    df = pd.DataFrame()

In [33]:
# Реализация метода Search-Ask

def num_tokens(text: str, model: str = GPT_MODEL) -> int:
    """Подсчитывает количество токенов в строке для указанной модели."""
    tiktoken_model = model.split("/")[-1]  # openai/gpt-4o-mini -> gpt-4o-mini
    encoding = tiktoken.encoding_for_model(tiktoken_model)
    return len(encoding.encode(text))


def get_embedding(text: str, model: str = EMBEDDING_MODEL) -> list[float]:
    """Получает вектор эмбеддинга для текста через OpenAI Embedding API."""
    response = openai_client.embeddings.create(input=[text], model=model)
    return response.data[0].embedding


def strings_ranked_by_relatedness(
    query: str,
    df: pd.DataFrame,
    relatedness_fn= lambda x, y: 1 - spatial.distance.cosine(x, y),
    top_n: int = 100
) -> tuple[list[str], list[float]]:
    """
    Ранжирует секции текста по релевантности запросу через косинусную близость.
    Возвращает top_n наиболее релевантных строк и их оценки.
    """
    query_embedding = get_embedding(query)
    strings_and_relatednesses = [
        (row["text"], relatedness_fn(query_embedding, row["embedding"]))
        for i, row in df.iterrows()
    ]
    strings_and_relatednesses.sort(key=lambda x: x[1], reverse=True)
    strings, relatednesses = zip(*strings_and_relatednesses)
    return strings[:top_n], relatednesses[:top_n]


def query_message(
    query: str,
    df: pd.DataFrame,
    model: str,
    token_budget: int
) -> str:
    """
    Формирует сообщение с наиболее релевантными статьями в качестве контекста для GPT.
    Прекращает добавление статей при превышении лимита токенов.
    """
    strings, _ = strings_ranked_by_relatedness(query, df)
    message = (
        'Use the below articles on the 2022 Winter Olympics to answer '
        'the subsequent question. If the answer cannot be found in the '
        'articles, write "I could not find an answer."'
    )
    question = f"\n\nQuestion: {query}"
    for string in strings:
        next_article = f'\n\nWikipedia article section:\n"""\n{string}\n"""'
        if num_tokens(message + next_article + question, model=model) > token_budget:
            break
        else:
            message += next_article
    return message + question


def ask(
    query: str,
    df: pd.DataFrame,
    model: str = GPT_MODEL,
    token_budget: int = 4096 - 500,
    print_message: bool = False
) -> str:
    """
    Отвечает на вопрос, используя метод Search-Ask.
    Ищет в базе знаний релевантные секции, затем отправляет запрос GPT.
    """
    message = query_message(query, df, model=model, token_budget=token_budget)
    if print_message:
        print(message)
    messages = [
        {"role": "system", "content": "You answer questions about the 2022 Winter Olympics."},
        {"role": "user", "content": message},
    ]
    response = openai_client.chat.completions.create(
        model=model, messages=messages, temperature=0
    )
    return response.choices[0].message.content

In [10]:
# Инициализация бота и определение обработчиков

bot = Bot(token=B_TOKEN)
dp = Dispatcher()


@dp.message(Command("start"))
async def cmd_start(message: types.Message) -> None:
    """Обрабатывает команду /start."""
    await message.answer(
        "Привет! Я бот, который знает о зимних Олимпийских играх 2022.\n"
        "Задай мне любой вопрос об Олимпиаде, и я отвечу на основе моей базы знаний.\n"
        "Используй /help, чтобы узнать подробнее."
    )


@dp.message(Command("help"))
async def cmd_help(message: types.Message) -> None:
    """Обрабатывает команду /help - возвращает информацию о базе знаний."""
    info = (
        "Информация о базе знаний:\n"
        f"- Тематика: 2022 Winter Olympics\n"
        f"- Количество записей: {len(df)}\n"
        f"- Пример запроса: What were the medal counts for the 2022 Winter Olympics?"
    )
    await message.answer(info)


@dp.message()
async def handle_question(message: types.Message) -> None:
    """Обрабатывает любое текстовое сообщение как вопрос к базе знаний."""
    if message.text and message.text.startswith("/"):
        return
    await message.answer("Ищу ответ в базе знаний...")
    try:
        answer = ask(message.text, df)
        await message.answer(answer)
    except Exception as e:
        logger.exception("Ошибка при обработке вопроса")
        await message.answer(f"Извините, произошла ошибка: {e}")

In [35]:
# Запуск поллинга
async def main() -> None:
    """Запускает цикл поллинга бота."""
    await dp.start_polling(bot)

asyncio.run(main())

ERROR:__main__:Ошибка при обработке вопроса
Traceback (most recent call last):
  File "/tmp/ipykernel_1488/4115517107.py", line 36, in handle_question
    answer = ask(message.text, df)
             ^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_1488/1485527601.py", line 72, in ask
    message = query_message(query, df, model=model, token_budget=token_budget)
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_1488/1485527601.py", line 45, in query_message
    strings, _ = strings_ranked_by_relatedness(query, df)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_1488/1485527601.py", line 25, in strings_ranked_by_relatedness
    query_embedding = get_embedding(query)
                      ^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_1488/1485527601.py", line 11, in get_embedding
    response = openai_client.embeddings.create(input=[text], model=model)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^